In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "vlamings2006great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Vlamings-Uher-Call_2006_reversed-contingency_I_raw-data.sav",)

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)

df['study_id']="vlamings2006great"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns


In [3]:
df.rename(columns={"name": "ape",
    "species":"species_original",
    "conditio":"condition"}, inplace=True)

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')


In [5]:
df['date'] = df['date'].astype(str)
df[['day_month', 'year']] = df['date'].str.split('.',expand=True)
df['year'] = '20' + df['year'].astype(str)
df['day_month'] = df['day_month'].astype(str).str.pad(4, 'left', '0')
df['day'] = df['day_month'].str.slice(0,2)
df['month'] = df['day_month'].str.slice(2,4)

df['year'].replace('2093', '2003', inplace=True, regex=True)

In [6]:
df.replace(':', '_', inplace=True, regex=True)
df.replace('vs.', 'vs', inplace=True, regex=True)

condition_rename = [['color_ zero vs four raisins','covered_0-4'],
                     ['no color_ one vs four raisins','visible_1-4'],
                     ['no color_ zero vs four raisins', 'visible_0-4'] ,
                     ['color_ one vs four raisins','covered_1-4']]
for x,y in condition_rename:
    df['condition'].replace(x, y, inplace=True)

In [7]:
response_rename = [['black dish','covered_black_dish'],
    ['no color dish containing one raisin', 'visible_1_raisin'],
    ['no color dish containing zero raisins','visible_0_raisins'],
    ['no color dish containing four raisins', 'visible_4_raisins'],
    ['orange dish','covered_orange_dish'],
    ['purple dish','covered_purple_dish']]

for x,y in response_rename:
    df['choice'].replace(x, y, inplace=True)
    df['left'].replace(x, y, inplace=True)
    df['right'].replace(x, y, inplace=True)


In [8]:

# df.columns
df.rename(columns={"ape": "participant"}, inplace=True)

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365 

In [9]:
vlamings2006great_standardized=df[['study_id', 'year', 'month', 'day',  
         'participant',  'age_in_years','sex','species', 'session', 'trial', 'condition',
       'left', 'right', 'choice', 'corr' ]]
comp_out_path_stand = os.path.join(out_pathway, 'vlamings2006great_standardized.csv')
vlamings2006great_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =vlamings2006great_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
vlamings2006great_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'vlamings2006great_glossary.csv')
vlamings2006great_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

